# H-002 Baseline Experiment

WP-2c baseline notebook. This notebook reproduces the baseline metrics from repo-truth data and helpers. It does not append to `research/trial_log.jsonl` when rerun. The committed trial ID is `357cca8c-d560-4d43-9037-b8bc24784a84`.

Evaluation note: no separate IS/OOS split convention exists in repo truth. The baseline uses the full registered evaluation range for both `in_sample_sharpe` and `oos_sharpe`, matching the current runbook baseline pattern.

Gate 2 semantics note: this notebook reports Gate 2 according to current `lib/evaluation.py` repo truth. The displayed legacy `GateReport` wording may read `DSR > 1.0 at 95% CI`, but the evaluator currently checks whether `dsr_val` exceeds 0.95 against its expected maximum-SR benchmark from the supplied trial count. Interpret the Gate 2 result according to the evaluator implementation, not as proof that the strategy met a true Sharpe > 1.0 standard.

In [1]:
from pathlib import Path
import json
import math
import pandas as pd
from lib.backtest import run_backtest
from lib.evaluation import check_promotion_gates, cost_drag, hit_rate, max_drawdown, profit_factor, sharpe

OHLCV_PATH = Path('data/processed/binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet')
FUNDING_PATH = Path('data/processed/binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet')

def load_ohlcv():
    df = pd.read_parquet(OHLCV_PATH).copy()
    df.index = pd.to_datetime(df['close_time_ms'], unit='ms', utc=True)
    return df[['open', 'high', 'low', 'close', 'volume']].astype(float).sort_index()

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df['fundingTime'].astype('int64'), unit='ms', utc=True)
    return df.rename(columns={'fundingRate': 'funding_rate'})[['funding_rate']].astype(float).sort_index()

def h001_signal(ohlcv, funding, lookback=6, threshold=0.0001):
    lb = int(round(lookback))
    aligned = funding['funding_rate'].reindex(ohlcv.index.union(funding.index)).sort_index().ffill().reindex(ohlcv.index)
    rolling_mean = aligned.rolling(lb, min_periods=lb).mean()
    signal = pd.Series(0.0, index=ohlcv.index, name='signal')
    signal[rolling_mean > threshold] = -1.0
    signal[rolling_mean < -threshold] = 1.0
    signal[rolling_mean.isna()] = 0.0
    return signal

def h002_signal(ohlcv, lookback=20):
    lb = int(round(lookback))
    upper = ohlcv['high'].shift(1).rolling(lb, min_periods=lb).max()
    lower = ohlcv['low'].shift(1).rolling(lb, min_periods=lb).min()
    signal = pd.Series(0.0, index=ohlcv.index, name='signal')
    current = 0.0
    for ts in ohlcv.index:
        if pd.isna(upper.loc[ts]) or pd.isna(lower.loc[ts]):
            current = 0.0
        elif ohlcv.loc[ts, 'close'] > upper.loc[ts]:
            current = 1.0
        elif ohlcv.loc[ts, 'close'] < lower.loc[ts]:
            current = -1.0
        signal.loc[ts] = current
    return signal

def run_candidate(signal, ohlcv, funding):
    return run_backtest(
        signal=signal,
        ohlcv=ohlcv,
        funding=funding,
        fee_bps_taker=4.0,
        fee_bps_maker=2.0,
        slippage_bps=2.0,
        initial_equity=10000.0,
        execution='next_bar_open',
        stream='klines',
        interval='4h',
    )

def summarize(result):
    returns = result.equity_curve.pct_change().dropna()
    mdd, peak, trough = max_drawdown(result.equity_curve)
    cd = cost_drag(result)
    return {
        'final_equity': float(result.equity_curve.iloc[-1]),
        'total_pnl': float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]),
        'sharpe': float(sharpe(returns, '4h')),
        'max_drawdown': float(mdd),
        'n_obs': int(len(returns)),
        'n_trades': int(len(result.trades_df)),
        'hit_rate': float(hit_rate(result.trades_df)),
        'profit_factor': float(profit_factor(result.trades_df)),
        'fees_paid': float(result.fees_paid),
        'slippage_paid': float(result.slippage_paid),
        'funding_paid': float(result.funding_paid),
        'gross_pnl': float(cd['gross_pnl']),
        'cost_drag_fraction': float(cd['cost_drag_fraction']) if math.isfinite(cd['cost_drag_fraction']) else None,
        'skipped_bars': int(result.skipped_bars),
    }

def sensitivity_h001(ohlcv, funding, base_sharpe):
    variants = [
        ('lookback_up_25pct', {'lookback': 7.5, 'threshold': 0.0001}),
        ('lookback_down_25pct', {'lookback': 4.5, 'threshold': 0.0001}),
        ('threshold_up_25pct', {'lookback': 6, 'threshold': 0.000125}),
        ('threshold_down_25pct', {'lookback': 6, 'threshold': 0.000075}),
    ]
    rows = []
    for name, params in variants:
        sr = summarize(run_candidate(h001_signal(ohlcv, funding, **params), ohlcv, funding))['sharpe']
        rows.append({'variant': name, 'sharpe': sr, 'ratio_to_base': sr / base_sharpe if base_sharpe != 0 else 0.0})
    return rows

def sensitivity_h002(ohlcv, funding, base_sharpe):
    rows = []
    for name, params in [('lookback_up_25pct', {'lookback': 25}), ('lookback_down_25pct', {'lookback': 15})]:
        sr = summarize(run_candidate(h002_signal(ohlcv, **params), ohlcv, funding))['sharpe']
        rows.append({'variant': name, 'sharpe': sr, 'ratio_to_base': sr / base_sharpe if base_sharpe != 0 else 0.0})
    return rows

ohlcv = load_ohlcv()
funding = load_funding()
signal = h002_signal(ohlcv, lookback=20)
result = run_candidate(signal, ohlcv, funding)
metrics = summarize(result)
sensitivity = sensitivity_h002(ohlcv, funding, metrics['sharpe'])
gate_report = check_promotion_gates(
    result=result,
    trial_count=1,
    params_meta={'n_free_params': 1, 'sensitivity_min_ratio': min(row['ratio_to_base'] for row in sensitivity), 'is_sharpe': metrics['sharpe']},
    freq='4h',
)
print(json.dumps({
    'hypothesis_id': 'H-002',
    'metrics': metrics,
    'sensitivity': sensitivity,
    'gate_report': gate_report.to_dict(),
}, indent=2, sort_keys=True))


{
  "config_sha256": "862786f6b10e80dc7955ef59fdc4ef867700973529a735c13ba82e6b2d2500f5",
  "dataset_ids": [
    "binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z"
  ],
  "dataset_sha256": "9c52bdd9b90a5c4c3f3857ee39de6941f0bd6de038007dc712616f5d6618fb87",
  "gate_report": {
    "gates": [
      {
        "gate_id": 1,
        "gate_name": "OOS Sharpe >= 1.0",
        "note": "",
        "passed": false,
        "threshold": 1.0,
        "value": 0.0758
      },
      {
        "gate_id": 2,
        "gate_name": "DSR > 1.0 at 95% CI",
        "note": "P(SR>1.0)=1.0000; SR benchmark=0.0000",
        "passed": true,
        "threshold": 1.0,
        "value": 1.0
      },
      {
        "gate_id": 3,
        "gate_name": "Per-fold max DD <= 20%",
        "note": "",
        "passed": false,
        "threshold": 0.2,
        "value": 1.205
      },
      {
        "gate_id": 4,
        "gate_n